# Pipelines B (LLM-only) & C (Hybrid) — v2: executed verification notebook
**What runs in this kernel:** fresh 5-fold CV for B and C on the cached `features_B.parquet` / `features_C.parquet` (307,511 customers each),
plus live predictions from the saved `model_B.pkl` / `model_C.pkl`.
**What is loaded, not regenerated:** the LLM features (6 recorded Qwen calls, `qwen/qwen3.8-27b`, responses in `outputs/llm_logs/`), the 8-stage gate
results (`outputs/validation_log.csv`), the matrices and the final models. No Groq calls are made here.

In [1]:
import sys, os, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('BFSI_THREADS', '8')
import numpy as np, pandas as pd, joblib, pyarrow.parquet as pq
from src import config as C, train
from sklearn.metrics import roc_auc_score
RES = json.loads((C.OUT_DIR/'final_results.json').read_text())
print('ROOT', ROOT, '| packaged run finished', RES['run']['finished_at_local'], '| status', RES['execution_status'])

ROOT /workspace/bfsi-mtp/BFSI_MTP_Demo | packaged run finished 2026-11-11 23:11:45 Asia/Kolkata (UTC+05:30) | status complete


In [2]:
# PROVENANCE of the cached feature matrices (LOADED, not regenerated here)
for P in ['B', 'C']:
    print(P, 'features_%s.parquet sha256' % P, RES['artifacts_sha256']['features'][P][:16], '| model pkl sha256', RES['artifacts_sha256']['models'][P]['pkl'][:16])
print('input CSV sha256 recorded for', len(RES['data']['sha256']), 'files; generation log: outputs/run_log.txt')

B features_B.parquet sha256 3dad2885d003dee2 | model pkl sha256 6b365a648209b8ff
C features_C.parquet sha256 f7bf554c2109b5af | model pkl sha256 ca3e99ebf105c04a
input CSV sha256 recorded for 9 files; generation log: outputs/run_log.txt


In [3]:
# LLM / gate provenance (LOADED)
print('model', RES['llm']['model'], '| calls', len(RES['llm']['calls']), '| gate', RES['gate'])
vl = pd.read_csv(C.OUT_DIR/'validation_log.csv'); display(vl[['feature_name','source_group','failure_type','failure_detail']])

model qwen/qwen3.8-27b | calls 6 | gate {'proposed': 32, 'passed': 26, 'failed': 6, 'by_type': {'passed': 26, 'constant': 5, 'security_violation': 1}}


,feature_name,source_group,failure_type,failure_detail
0,social_circle_def_30_rate,application,passed,NaN
1,social_circle_def_60_rate,application,passed,NaN
2,ext_source_dispersion,application,passed,NaN
3,address_mismatch_count,application,passed,NaN
4,enquiry_acceleration_ratio,application,passed,NaN
5,building_age_vs_client_age,application,passed,NaN
6,bureau_active_credit_count,bureau,passed,NaN
7,bureau_debt_to_limit_ratio,bureau,passed,NaN
8,bureau_max_overdue_amount,bureau,passed,NaN
9,bureau_balance_dpd30_plus_count,bureau,passed,NaN


## Full rebuild (NOT executed in this notebook)
The feature matrices were generated by `python run_all.py BC` (logged in `outputs/run_log.txt`) using the cached LLM responses.
To regenerate from raw Kaggle data run in a terminal: `python run_all.py BC` (then `python run_all.py audit` and `python run_all.py finalize`).

## Pipeline B — fresh 5-fold CV (trained here)

In [4]:
# FRESH 5-fold CV executed in THIS kernel on the cached features_B.parquet (ephemeral models, nothing saved)
X = pq.read_table(C.FEAT_DIR/'features_B.parquet').to_pandas()
sc = pd.read_parquet(C.OUT_DIR/'scored_table.parquet', columns=['SK_ID_CURR','TARGET','oof_B','pred_B'])
assert np.array_equal(X['SK_ID_CURR'].to_numpy(), sc['SK_ID_CURR'].to_numpy()) and len(X) == 307511
ids = X.pop('SK_ID_CURR'); y = sc['TARGET'].astype(int)
t0 = time.time(); oof, m, _ = train.run_cv(X, y, 'B')
print(f'fresh CV time {time.time()-t0:.0f}s')
ref = RES['pipelines']['B']
print('fresh OOF AUC', m['oof_auc'], '| packaged', ref['oof_auc'], '| fresh folds', m['fold_aucs'])
print('max |fresh OOF - packaged oof_B| =', float(np.max(np.abs(oof - sc['oof_B'].to_numpy()))))

  B fold 1 AUC=0.76906 trees=375 t=6s


  B fold 2 AUC=0.77807 trees=360 t=13s


  B fold 3 AUC=0.77278 trees=367 t=20s


  B fold 4 AUC=0.77827 trees=375 t=27s


  B fold 5 AUC=0.76996 trees=374 t=33s


  B OOF AUC=0.77360 KS=0.41214 cv_std=0.00391
fresh CV time 33s
fresh OOF AUC 0.7736 | packaged 0.7736 | fresh folds [0.76906, 0.77807, 0.77278, 0.77827, 0.76996]
max |fresh OOF - packaged oof_B| = 0.0


## Pipeline B — final model (loaded)

In [5]:
# LOADED final model_B.pkl (trained by run_all.py) -> live predictions vs scored table
model = joblib.load(C.MODEL_DIR/'model_B.pkl')
assert list(model.booster_.feature_name()) == list(X.columns)
p = model.predict_proba(X)[:, 1]
print('model_B: full-matrix live predict max |diff| vs pred_B =', float(np.max(np.abs(p - sc['pred_B'].to_numpy()))))
demo = sc.sample(5, random_state=1).assign(live_pred=lambda d: model.predict_proba(X.loc[d.index])[:, 1])
display(demo[['SK_ID_CURR','TARGET','pred_B','live_pred']])
del X; import gc; gc.collect()

model_B: full-matrix live predict max |diff| vs pred_B = 0.0


0

## Pipeline C — fresh 5-fold CV (trained here)

In [6]:
# FRESH 5-fold CV executed in THIS kernel on the cached features_C.parquet (ephemeral models, nothing saved)
X = pq.read_table(C.FEAT_DIR/'features_C.parquet').to_pandas()
sc = pd.read_parquet(C.OUT_DIR/'scored_table.parquet', columns=['SK_ID_CURR','TARGET','oof_C','pred_C'])
assert np.array_equal(X['SK_ID_CURR'].to_numpy(), sc['SK_ID_CURR'].to_numpy()) and len(X) == 307511
ids = X.pop('SK_ID_CURR'); y = sc['TARGET'].astype(int)
t0 = time.time(); oof, m, _ = train.run_cv(X, y, 'C')
print(f'fresh CV time {time.time()-t0:.0f}s')
ref = RES['pipelines']['C']
print('fresh OOF AUC', m['oof_auc'], '| packaged', ref['oof_auc'], '| fresh folds', m['fold_aucs'])
print('max |fresh OOF - packaged oof_C| =', float(np.max(np.abs(oof - sc['oof_C'].to_numpy()))))

  C fold 1 AUC=0.78297 trees=375 t=12s


  C fold 2 AUC=0.78993 trees=375 t=25s


  C fold 3 AUC=0.78416 trees=375 t=37s


  C fold 4 AUC=0.78962 trees=372 t=49s


  C fold 5 AUC=0.78241 trees=375 t=60s
  C OOF AUC=0.78580 KS=0.4319 cv_std=0.00328
fresh CV time 60s
fresh OOF AUC 0.7858 | packaged 0.7858 | fresh folds [0.78297, 0.78993, 0.78416, 0.78962, 0.78241]
max |fresh OOF - packaged oof_C| = 0.0


## Pipeline C — final model (loaded)

In [7]:
# LOADED final model_C.pkl (trained by run_all.py) -> live predictions vs scored table
model = joblib.load(C.MODEL_DIR/'model_C.pkl')
assert list(model.booster_.feature_name()) == list(X.columns)
p = model.predict_proba(X)[:, 1]
print('model_C: full-matrix live predict max |diff| vs pred_C =', float(np.max(np.abs(p - sc['pred_C'].to_numpy()))))
demo = sc.sample(5, random_state=1).assign(live_pred=lambda d: model.predict_proba(X.loc[d.index])[:, 1])
display(demo[['SK_ID_CURR','TARGET','pred_C','live_pred']])
del X; import gc; gc.collect()

model_C: full-matrix live predict max |diff| vs pred_C = 0.0


23

In [8]:
pd.DataFrame(RES['pipelines']).T[['n_features','oof_auc','ks','cv_std']]

,n_features,oof_auc,ks,cv_std
A,241,0.78362,0.42897,0.00349
B,191,0.7736,0.41214,0.00391
C,289,0.7858,0.4319,0.00328
